# 🎭 08Comedy 單口喜劇 — 自動訓練管線

## 本筆記本功能
1. **自動過濾**：從 `@08comedy` 頻道找出「單口喜劇」相關影片
2. **完整下載**：下載影片 (MP4) + 音訊 (WAV) + 字幕 (JSON3)
3. **特徵擷取**：笑聲偵測、表情分析、字幕對齊
4. **即存即刪**：每部處理完立即刪除原始影片，節省空間
5. **訓練模型**：Reward Model 幽默分數預測

## ⚠️ 空間估計（完整 MP4）
| 單部影片 | 暫存空間 |
|---------|--------|
| 30 分鐘影片 (720p MP4) | ~500 MB |
| WAV 音軌 (16kHz) | ~55 MB |
| 字幕 JSON | <1 MB |
| **峰值（同時存在）** | **~600 MB** |
| 處理後保留（特徵數據）| **~3 MB** |

> ✅ Colab 有 ~80GB 空間，處理完即刪，絕對夠用

## 🔧 STEP 0 — 環境初始化（每次 Colab 重啟都要跑）

In [ ]:
# ── 0-A. 掛載 Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/humor_bot_08comedy'
DRIVE_DATASET = f'{DRIVE_ROOT}/dataset'
DRIVE_MODEL = f'{DRIVE_ROOT}/models'
DRIVE_LOGS = f'{DRIVE_ROOT}/logs'

for d in [DRIVE_ROOT, DRIVE_DATASET, DRIVE_MODEL, DRIVE_LOGS]:
    os.makedirs(d, exist_ok=True)

print(f'✅ Google Drive 掛載成功')
print(f'   資料集存放: {DRIVE_DATASET}')
print(f'   模型存放: {DRIVE_MODEL}')

In [ ]:
# ── 0-B. 防 Idle Timeout（雙重機制）───────────────────────────────
from IPython.display import Javascript, display
import threading, time, subprocess

# 1. JavaScript：每 50 秒模擬使用者動作
display(Javascript('''
if (window._keepAliveInterval) clearInterval(window._keepAliveInterval);
window._keepAliveInterval = setInterval(() => {
    document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true}));
    console.log('[KeepAlive]', new Date().toLocaleTimeString());
}, 45000);
console.log('[KeepAlive] 已啟動');
'''))

# 2. Python：背景執行緒每 55 秒呼叫 nvidia-smi
_ka_stop = threading.Event()

def _keepalive():
    while not _ka_stop.is_set():
        r = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.free',
                            '--format=csv,noheader,nounits'],
                           capture_output=True, text=True)
        info = r.stdout.strip() if r.returncode == 0 else 'CPU only'
        print(f'[{time.strftime("%H:%M:%S")}] GPU: {info} | '
              f'Disk: {_disk_gb():.1f}GB used')
        _ka_stop.wait(55)

def _disk_gb(path='/content'):
    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(path) for f in files
        if not os.path.islink(os.path.join(r, f))
    )
    return total / 1e9

_ka_thread = threading.Thread(target=_keepalive, daemon=True)
_ka_thread.start()
print('✅ 防 Timeout 已啟動（JS + Python 雙重保障）')
print('   停止: _ka_stop.set()')

In [ ]:
# ── 0-C. 安裝套件 ────────────────────────────────────────────────
print('📦 安裝套件中...')
!pip install -q yt-dlp faster-whisper soundfile librosa
!pip install -q tensorflow tensorflow-hub
!pip install -q sentence-transformers transformers datasets accelerate peft trl
!pip install -q opencv-python-headless Pillow
!pip install -q deepface  # 表情辨識（用來分析觀眾/演員表情）
!pip install -q chromadb openai
!pip install -q numpy pandas scipy matplotlib tqdm
!apt-get install -qq -y ffmpeg > /dev/null 2>&1
print('✅ 安裝完成！')

In [ ]:
# ── 0-D. Clone 你的 GitHub 專案 ─────────────────────────────────
import sys

PROJECT_DIR = '/content/humor'
SRC_DIR = f'{PROJECT_DIR}/src'

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/1122-gggggg/humor.git {PROJECT_DIR}
else:
    # 如果已存在，pull 最新版
    !cd {PROJECT_DIR} && git pull

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'✅ 專案已就緒: {PROJECT_DIR}')
print(f'   Python 路徑: {SRC_DIR}')

## 🎯 STEP 1 — 從 @08comedy 過濾單口喜劇影片

In [ ]:
# ── 1. 抓取頻道影片清單並用關鍵字過濾「單口喜劇」────────────────────
import yt_dlp
import json, re
from pathlib import Path

CHANNEL_URL = 'https://www.youtube.com/@08comedy/videos'
MAX_VIDEOS_TO_SCAN = 200   # 掃描最近幾部（不是全部下載，只是掃清單）

# 單口喜劇關鍵字（包含正面關鍵字 & 排除關鍵字）
INCLUDE_KEYWORDS = [
    '單口喜劇', '脫口秀', 'stand-up', 'standup', 'stand up',
    '開放麥', 'open mic', '段子', '演出', '表演',
    '喜劇之夜', '喜劇人', '笑點', '搞笑',
]

EXCLUDE_KEYWORDS = [
    '幕後', 'behind the scene', '採訪', 'interview',
    '廣告', 'trailer', '預告', '短片', 'short',
    '教學', 'tutorial', 'vlog', 'podcast', '訪談',
]

def is_standup_video(title: str, description: str = '') -> tuple[bool, str]:
    """判斷是否為單口喜劇影片，回傳 (是否符合, 匹配關鍵字)"""
    text = f'{title} {description}'.lower()

    # 先排除
    for kw in EXCLUDE_KEYWORDS:
        if kw.lower() in text:
            return False, f'排除關鍵字: {kw}'

    # 再包含
    for kw in INCLUDE_KEYWORDS:
        if kw.lower() in text:
            return True, kw

    return False, '無匹配關鍵字'


print(f'🔍 正在掃描頻道: {CHANNEL_URL}')
print(f'   關鍵字: {INCLUDE_KEYWORDS[:5]}...')

ydl_opts = {
    'extract_flat': True,
    'quiet': True,
    'playlistend': MAX_VIDEOS_TO_SCAN,
    'ignoreerrors': True,
}

standup_videos = []
all_videos = []

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(CHANNEL_URL, download=False)
    entries = info.get('entries', [])

    for entry in entries:
        if not entry:
            continue
        vid_id = entry.get('id', '')
        title = entry.get('title', '')
        description = entry.get('description', '')
        duration = entry.get('duration', 0) or 0
        url = f'https://www.youtube.com/watch?v={vid_id}'

        all_videos.append({'id': vid_id, 'title': title, 'duration': duration, 'url': url})

        # 過短的影片跳過（脫口秀通常 > 3 分鐘）
        if duration < 180:
            continue

        is_standup, reason = is_standup_video(title, description)
        if is_standup:
            standup_videos.append({
                'id': vid_id, 'title': title,
                'duration': duration, 'url': url,
                'match_reason': reason
            })

print(f'\n📊 掃描結果:')
print(f'   頻道總影片: {len(all_videos)} 部')
print(f'   ✅ 判斷為單口喜劇: {len(standup_videos)} 部')
print()
print('📋 篩選出的單口喜劇影片:')
for v in standup_videos:
    mins = v['duration'] // 60
    print(f"  [{mins}min] {v['title'][:50]} (匹配: {v['match_reason']})")

In [ ]:
# ── 1-B. 手動確認 & 微調清單 ─────────────────────────────────────
# 如果對上面的自動過濾結果不滿意，可以在這裡手動加入或移除

# 手動加入（貼上 YouTube URL 即可）
MANUAL_ADD_URLS = [
    # 'https://www.youtube.com/watch?v=XXXXXXXXX',
]

# 手動排除（貼上 video ID）
MANUAL_EXCLUDE_IDS = [
    # 'XXXXXXXXX',
]

# 合併最終清單
final_urls = [v['url'] for v in standup_videos if v['id'] not in MANUAL_EXCLUDE_IDS]
final_urls += MANUAL_ADD_URLS
final_urls = list(dict.fromkeys(final_urls))  # 去重

print(f'🎯 最終訓練影片清單: {len(final_urls)} 部')
for i, url in enumerate(final_urls, 1):
    print(f'  {i:3d}. {url}')

# 儲存清單到 Drive（下次可直接讀取）
url_list_path = f'{DRIVE_ROOT}/standup_urls.txt'
with open(url_list_path, 'w', encoding='utf-8') as f:
    for url in final_urls:
        f.write(f'{url}\n')
print(f'\n💾 清單已儲存: {url_list_path}')

## ⬇️ STEP 2 — 完整影片下載 + 特徵擷取（串流模式）

In [ ]:
# ── 2-A. 下載工具：同時抓取 MP4 + WAV + 字幕 ────────────────────
import subprocess, shutil, traceback, dataclasses
from pathlib import Path

WORK_DIR = Path('/content/work')  # 暫存目錄（處理完即刪）
WORK_DIR.mkdir(exist_ok=True)

PROCESSED_LOG_PATH = f'{DRIVE_ROOT}/processed_ids.json'


def load_processed_ids():
    if os.path.exists(PROCESSED_LOG_PATH):
        with open(PROCESSED_LOG_PATH) as f:
            return set(json.load(f))
    return set()


def mark_processed(video_id):
    ids = load_processed_ids()
    ids.add(video_id)
    with open(PROCESSED_LOG_PATH, 'w') as f:
        json.dump(list(ids), f)
    # 同步到 Drive（確保即使 Colab 斷線也不會重複處理）
    shutil.copy2(PROCESSED_LOG_PATH, PROCESSED_LOG_PATH)


def download_full_video(url: str, work_dir: Path) -> dict | None:
    """
    下載完整影片 (MP4) + 音訊 (WAV 16kHz) + 字幕
    回傳路徑字典或 None（失敗時）
    """
    ydl_opts = {
        # 下載影片（最多 720p，避免 4K 佔滿空間）
        'format': 'bestvideo[height<=720][ext=mp4]+bestaudio/best[height<=720]',
        'merge_output_format': 'mp4',
        'outtmpl': str(work_dir / '%(id)s.%(ext)s'),

        # 同時下載字幕
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['zh-TW', 'zh-Hant', 'zh', 'en'],
        'subtitlesformat': 'json3',

        # 後處理：額外提取 WAV（用於 Whisper + YAMNet）
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
        }],
        'postprocessor_args': ['-ar', '16000', '-ac', '1'],
        'keepvideo': True,  # ← 重要：保留影片，不要讓 postprocessor 刪除它

        # 留言（用於人設分析）
        'getcomments': True,
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            vid_id = info['id']
            title = info.get('title', '')
            duration = info.get('duration', 0)

        # 確認下載的檔案
        video_path = work_dir / f'{vid_id}.mp4'
        audio_path = work_dir / f'{vid_id}.wav'

        # 如果 WAV 沒有被成功提取，用 ffmpeg 手動轉換
        if not audio_path.exists() and video_path.exists():
            print(f'  ⚙️  手動提取 WAV...')
            subprocess.run([
                'ffmpeg', '-y', '-i', str(video_path),
                '-ar', '16000', '-ac', '1', str(audio_path)
            ], capture_output=True)

        # 找字幕
        subtitle_path = None
        for lang in ['zh-TW', 'zh-Hant', 'zh', 'en']:
            p = work_dir / f'{vid_id}.{lang}.json3'
            if p.exists():
                subtitle_path = p
                break

        sizes = {
            'mp4': video_path.stat().st_size / 1e6 if video_path.exists() else 0,
            'wav': audio_path.stat().st_size / 1e6 if audio_path.exists() else 0,
        }

        print(f'  📦 下載完成: {title[:40]}')
        print(f'     MP4: {sizes["mp4"]:.0f} MB | WAV: {sizes["wav"]:.0f} MB | 字幕: {"✅" if subtitle_path else "❌"}')

        return {
            'id': vid_id,
            'title': title,
            'duration': duration,
            'video_path': str(video_path) if video_path.exists() else None,
            'audio_path': str(audio_path) if audio_path.exists() else None,
            'subtitle_path': str(subtitle_path) if subtitle_path else None,
            'info': info,
        }
    except Exception as e:
        print(f'  ❌ 下載失敗: {e}')
        return None


def cleanup_video_files(vid_id: str, work_dir: Path):
    """刪除工作目錄中所有與 vid_id 相關的檔案"""
    deleted = []
    for f in work_dir.glob(f'{vid_id}*'):
        try:
            f.unlink()
            deleted.append(f.name)
        except:
            pass
    if deleted:
        print(f'  🧹 清理: {deleted}')


print('✅ 下載工具函式已就緒')

In [ ]:
# ── 2-B. 載入 AI 模型（只做一次）────────────────────────────────
print('🤖 載入 AI 模型（首次需數分鐘下載...）')

from humor_bot.data_engine.youtube_downloader import YouTubeDownloader, SubtitleSource
from humor_bot.data_engine.laughter_detector import LaughterDetector
from humor_bot.data_engine.audio_analyzer import AudioAnalyzer
from humor_bot.data_engine.alignment import SetupPunchlineAligner
from humor_bot.data_engine.auto_annotator import AutoAnnotationPipeline
from humor_bot.data_engine.video_analyzer import VideoAnalyzer

# YouTubeDownloader 只用於轉錄（不下載，我們自己下載）
_downloader = YouTubeDownloader(
    output_dir='/content/work',
    whisper_model_size='large-v3',
)

detector   = LaughterDetector(confidence_threshold=0.7)
analyzer   = AudioAnalyzer()
aligner    = SetupPunchlineAligner(min_laughter_confidence=0.7)
annotator  = AutoAnnotationPipeline(
    enable_video=True,            # 開啟影像分析
    enable_technique_analysis=False  # 先關閉 LLM 分析（加快速度）
)
vid_analyzer = VideoAnalyzer(
    sample_interval_sec=1.5,      # 每 1.5 秒取一幀分析
    audience_roi=(0.0, 0.5, 1.0, 0.5)  # 假設觀眾在畫面下半部
)

print('✅ 所有模型載入完成！')

In [ ]:
# ── 2-C. 主管線：逐部影片串流處理 ───────────────────────────────
# 流程：下載(MP4+WAV+字幕) → 轉錄 → 笑聲偵測 → 影像分析 → 對齊 → 標註 → 存 Drive → 刪除

import logging
logging.basicConfig(level=logging.WARNING)  # 只顯示警告以上，減少雜訊

all_candidates = []  # 累積所有候選集

# 讀取已有的候選集（斷點續傳）
candidates_path = f'{DRIVE_ROOT}/candidates.json'
if os.path.exists(candidates_path):
    with open(candidates_path) as f:
        all_candidates = json.load(f)
    print(f'📂 載入已有候選集: {len(all_candidates)} 筆')

already_done = load_processed_ids()
print(f'📋 斷點續傳：已跳過 {len(already_done)} 部已處理影片')

total = len(final_urls)
ok, skip, fail = 0, 0, 0

for i, url in enumerate(final_urls, 1):
    # 提取 video ID
    m = re.search(r'[?&]v=([a-zA-Z0-9_-]{11})', url)
    vid_preview = m.group(1) if m else '?'

    print(f'\n{"="*65}')
    print(f'[{i}/{total}] {url}')
    print(f'       磁碟使用: {_disk_gb():.1f} GB')

    if vid_preview in already_done:
        print('  ⏭️  已處理，跳過')
        skip += 1
        continue

    # 磁碟安全閥：若超過 65GB 強制清空暫存
    if _disk_gb() > 65:
        print('  ⚠️  磁碟超過 65 GB，強制清空 /content/work')
        shutil.rmtree(str(WORK_DIR), ignore_errors=True)
        WORK_DIR.mkdir(exist_ok=True)

    t0 = time.time()
    video_work_dir = WORK_DIR / vid_preview
    video_work_dir.mkdir(exist_ok=True)

    try:
        # ──── A. 下載完整影片 ─────────────────────────────────────
        print('  ⬇️  A. 下載 MP4 + WAV + 字幕...')
        dl = download_full_video(url, video_work_dir)
        if not dl:
            fail += 1
            shutil.rmtree(str(video_work_dir), ignore_errors=True)
            continue

        vid_id = dl['id']
        audio_path  = dl['audio_path']
        video_path  = dl['video_path']
        sub_path    = dl['subtitle_path']

        if not audio_path or not os.path.exists(audio_path):
            print('  ❌  WAV 不存在，跳過')
            fail += 1
            shutil.rmtree(str(video_work_dir), ignore_errors=True)
            continue

        # ──── B. 轉錄（字幕 or Whisper）──────────────────────────
        print('  🎙️  B. 語音轉錄...')
        if sub_path:
            segments = _downloader._parse_json3_subtitle(sub_path)
            src = '字幕'
        else:
            segments = _downloader._whisper_transcribe(audio_path)
            src = 'Whisper'
        print(f'     {src}: {len(segments)} 段')

        transcript_data = {
            'metadata': {'video_id': vid_id, 'subtitle_source': src},
            'segments': [dataclasses.asdict(s) for s in segments]
        }

        # ──── C. 笑聲偵測（YAMNet）───────────────────────────────
        print('  😂  C. 笑聲偵測...')
        laughter_events = detector.detect(audio_path)
        laughter_dicts = [
            {'start': e.start, 'end': e.end, 'duration': e.duration,
             'confidence': e.confidence, 'event_class': e.event_class}
            for e in laughter_events
        ]
        print(f'     偵測到 {len(laughter_events)} 個笑聲事件')

        # ──── D. 音訊特徵分析 ─────────────────────────────────────
        audio_features = []
        for e in laughter_events:
            feat = analyzer.analyze_segment(audio_path, e.start, e.end)
            audio_features.append({'start': feat.start, 'end': feat.end,
                                    'rms_db': feat.rms_db})

        # ──── E. 影像分析（觀眾表情）─────────────────────────────
        video_reactions = None
        if video_path and os.path.exists(video_path):
            print('  🎬  E. 分析影像（觀眾表情）...')
            try:
                result_v = vid_analyzer.analyze_video(video_path)
                video_reactions = [
                    {'timestamp': r.timestamp, 'positive_ratio': r.positive_ratio,
                     'happy_ratio': r.happy_ratio, 'surprise_ratio': r.surprise_ratio}
                    for r in result_v.reactions
                ]
                print(f'     分析 {result_v.total_frames_analyzed} 幀，'
                      f'平均正面比例: {result_v.avg_positive_ratio:.1%}')
            except Exception as ve:
                print(f'     ⚠️  影像分析失敗: {ve}')

        # ──── F. 對齊 Setup-Punchline ─────────────────────────────
        print('  📎  F. 對齊段子...')
        aligned_jokes = aligner.align(
            vid_id, transcript_data, laughter_dicts, audio_features
        )
        print(f'     對齊到 {len(aligned_jokes)} 個段子')

        # ──── G. 自動標註 ─────────────────────────────────────────
        print('  🏷️  G. 自動標註...')
        candidates = annotator.run(
            video_id=vid_id,
            aligned_jokes=aligned_jokes,
            laughter_events=laughter_dicts,
            audio_features=audio_features,
            video_reactions=video_reactions,
        )
        funny_cnt = sum(1 for c in candidates if c.auto_label == 'funny')
        print(f'     候選集: {len(candidates)} 筆 (😂 {funny_cnt} 個好笑)')

        # ──── H. 儲存到 Drive ─────────────────────────────────────
        new_dicts = [dataclasses.asdict(c) for c in candidates]
        all_candidates.extend(new_dicts)

        # 儲存累積候選集
        with open(candidates_path, 'w', encoding='utf-8') as f:
            json.dump(all_candidates, f, ensure_ascii=False, indent=2)

        # 單部影片的獨立結果（方便 debug）
        vid_out = f'{DRIVE_DATASET}/{vid_id}.json'
        with open(vid_out, 'w', encoding='utf-8') as f:
            json.dump({'video_id': vid_id, 'title': dl['title'],
                       'candidates': new_dicts}, f, ensure_ascii=False, indent=2)

        # ──── I. 刪除影片原始檔（節省空間）──────────────────────
        print('  🧹  I. 刪除原始影片...')
        shutil.rmtree(str(video_work_dir), ignore_errors=True)

        mark_processed(vid_id)
        ok += 1
        print(f'  ✅  耗時 {time.time()-t0:.0f}s | 累積候選集: {len(all_candidates)} 筆')

    except KeyboardInterrupt:
        print('\n⛔ 中斷。已儲存進度。')
        break
    except Exception as e:
        print(f'  ❌ 處理失敗: {e}')
        traceback.print_exc()
        fail += 1
        shutil.rmtree(str(video_work_dir), ignore_errors=True)

print(f'\n{"="*65}')
print(f'🎉 管線完成！✅ {ok} 成功 | ⏭️ {skip} 跳過 | ❌ {fail} 失敗')
print(f'📊 總候選集: {len(all_candidates)} 筆')
print(f'💾 儲存位置: {candidates_path}')

## 🤖 STEP 3 — 訓練 Reward Model

In [ ]:
# ── 3. 訓練 Reward Model（幽默分類器）───────────────────────────
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

# 載入最新候選集
with open(candidates_path, 'r', encoding='utf-8') as f:
    all_candidates = json.load(f)

valid = [
    c for c in all_candidates
    if c.get('setup_text','').strip() and c.get('punchline_text','').strip()
    and c.get('humor_score') is not None
]
print(f'📊 有效訓練資料: {len(valid)} 筆 (共 {len(all_candidates)} 筆)')

if len(valid) < 30:
    print('⚠️  資料太少，請先收集更多影片！')
else:
    MODEL_NAME = 'bert-base-chinese'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def preprocess(ex):
        texts = [f"{s} [SEP] {p}" for s, p in
                 zip(ex['setup_text'], ex['punchline_text'])]
        enc = tokenizer(texts, padding='max_length',
                        truncation=True, max_length=256)
        enc['labels'] = ex['humor_score']
        return enc

    ds = Dataset.from_list(valid).train_test_split(test_size=0.15, seed=42)
    tok_ds = ds.map(preprocess, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

    args = TrainingArguments(
        output_dir=f'{DRIVE_MODEL}/reward_ckpt',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=50,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        fp16=torch.cuda.is_available(),
        report_to='none',
        logging_steps=20,
    )

    def metrics(ep):
        p, l = ep
        p = p.squeeze()
        return {
            'mse': float(np.mean((p-l)**2)),
            'mae': float(np.mean(np.abs(p-l))),
            'pearson': float(np.corrcoef(p, l)[0,1]) if len(p)>1 else 0
        }

    trainer = Trainer(model=model, args=args,
                      train_dataset=tok_ds['train'],
                      eval_dataset=tok_ds['test'],
                      compute_metrics=metrics)

    print('🚀 開始訓練...')
    trainer.train()

    save_path = f'{DRIVE_MODEL}/reward_model_final'
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'✅ 模型已儲存: {save_path}')

## 🔮 STEP 4 — 即時預測：輸入段子文本 → 幽默分數

In [ ]:
# ── 4. 推論：輸入 setup + punchline → 預測觀眾反應 ───────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

REWARD_MODEL_PATH = f'{DRIVE_MODEL}/reward_model_final'

tok = AutoTokenizer.from_pretrained(REWARD_MODEL_PATH)
rm  = AutoModelForSequenceClassification.from_pretrained(REWARD_MODEL_PATH)
rm.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
_dev = next(rm.parameters()).device

def predict(setup: str, punchline: str) -> dict:
    inp = tok(f'{setup} [SEP] {punchline}',
              return_tensors='pt', truncation=True,
              max_length=256, padding=True).to(_dev)
    with torch.no_grad():
        score = float(torch.sigmoid(rm(**inp).logits[0][0]))
    label = '😂 觀眾很可能會大笑' if score>=0.7 else ('🙂 會心一笑' if score>=0.4 else '😑 可能冷場')
    return {'score': round(score, 3), 'prediction': label}

# 測試
tests = [
    ('我以前覺得婚姻是愛情的墳墓', '後來發現不對，是我的自由的墳墓'),
    ('我媽說我不懂感恩', '我說，媽，你生我的時候問過我嗎'),
    ('今天天氣很好', '適合出門走走'),
]

print('🎯 幽默預測結果:')
print('='*60)
for setup, punchline in tests:
    r = predict(setup, punchline)
    print(f'Setup:    {setup}')
    print(f'Punchline:{punchline}')
    print(f'→ Score: {r["score"]} | {r["prediction"]}')
    print('-'*40)